In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.linalg as linalg
import numpy as np
import random
from typing import List, Tuple

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
D = 5  # 维度
R = 1  # LoRA 秩
M = 10 # 数据点数量
SCALE = 1.0 # LoRA 缩放因子，简化为 1.0

# --- 2. 定义初始权重 W0 和目标更新 Delta_W_star ---
# W0: 5x5 对角矩阵 (作为预训练模型权重，冻结)
W0_np = np.eye(D, dtype=np.float32)
W0 = torch.tensor(W0_np)

# Delta_W_star: 目标秩一更新矩阵 (解析解)
u_np = np.array([[0.5]] * D, dtype=np.float32)  # 5x1 向量
v_T_np = np.array([[1.0, 0.0, 0.0, 0.0, 0.0]], dtype=np.float32)  # 1x5 向量

Delta_W_star = torch.tensor(u_np @ v_T_np)  # 5x5 秩一矩阵

# W_target: 目标权重
W_target = W0 + Delta_W_star



# --- 3. 定义微调训练集 St (10 条数据) ---
# 确保数据与您提供的具体数值一致
X_data_list = [
    [1.0, 0.5, -1.0, 0.2, -0.8], [-0.5, 1.0, 0.5, 1.0, -0.2], [0.2, -0.2, 1.5, 0.8, 0.1],
    [-1.2, 0.1, 0.3, -0.5, 1.2], [0.8, 1.2, -0.4, 0.3, 0.0], [-0.1, 0.9, 0.1, 0.6, -0.4],
    [0.6, 0.3, 0.9, 0.7, 1.1], [0.0, -1.0, 0.0, 0.5, -0.5], [-0.3, 0.8, 0.2, -0.1, 0.7],
    [1.5, -0.5, 0.5, 0.4, -0.3]
]
# 将列表转换为 M x D (10x5) 的张量
X = torch.tensor(X_data_list, dtype=torch.float32)

# 计算标签 Y = X @ W_target.T
# 注意：PyTorch/Numpy 默认是 X_i @ W，但我们的线性模型是 W x_i，所以这里使用 X @ W_target.T
# 或者更直观地，对每一行 x_i 进行 W_target @ x_i
Y = torch.stack([W_target @ x for x in X])

print("--- 训练集 St ---")
print(f"输入 X 形状: {X.shape} (10 条数据, 5 维特征)")
print(f"标签 Y 形状: {Y.shape} (10 条数据, 5 维输出)")
print("-" * 20)

# 确保 W0 不会参与梯度计算
W0.requires_grad_(False)

--- 训练集 St ---
输入 X 形状: torch.Size([10, 5]) (10 条数据, 5 维特征)
标签 Y 形状: torch.Size([10, 5]) (10 条数据, 5 维输出)
--------------------


tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])

In [ ]:
# --- 4. LoRA 矩阵初始化 ---
# B: 5 x 1 矩阵 (默认初始化为零)
B = nn.Parameter(torch.zeros(D, R))
# A: 1 x 5 矩阵 (默认初始化为小的随机高斯值)
A = nn.Parameter(torch.randn(R, D) * 0.01)

# --- 5. 设置优化器和损失函数 ---
# 仅优化 B 和 A 矩阵
optimizer = optim.Adam([B, A], lr=0.01)
criterion = nn.MSELoss() # 使用均方误差作为最小二乘损失的近似

# --- 6. 训练循环 ---
NUM_EPOCHS = 5000 # 训练较多轮次以保证收敛

for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()

    # 核心步骤：计算 W_updated = W0 + B @ A * SCALE
    # B (5x1) @ A (1x5) 得到 Delta_W (5x5)
    Delta_W_train = B @ A * SCALE
    W_updated = W0 + Delta_W_train

    # 计算预测 Y_pred = X @ W_updated.T
    # PyTorch/Numpy 的默认矩阵乘法是 (Batch x In) @ (In x Out) -> (Batch x Out)
    # 我们的模型是 W x_i，即 (5x5) @ (5x1)，所以 X @ W_updated.T
    Y_pred = X @ W_updated.T

    # 计算损失
    loss = criterion(Y_pred, Y)

    # 反向传播和优化
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        # 计算 Delta_W 的 Frobenius 范数误差
        with torch.no_grad():
             # 训练得到的 Delta_W 与目标 Delta_W_star 的差异
            delta_w_error = linalg.norm(Delta_W_train - Delta_W_star, ord='fro')
        print(f"Epoch {epoch+1:4d}/{NUM_EPOCHS}: Loss = {loss.item():.8f}, |ΔW_train - ΔW*|_F = {delta_w_error.item():.8f}")

print("-" * 20)
print("LoRA 训练完成。")

Epoch 1000/5000: Loss = 0.00000000, |ΔW_train - ΔW*|_F = 0.00000023
Epoch 2000/5000: Loss = 0.00000000, |ΔW_train - ΔW*|_F = 0.00001377
Epoch 3000/5000: Loss = 0.00000010, |ΔW_train - ΔW*|_F = 0.00094655
Epoch 4000/5000: Loss = 0.00000000, |ΔW_train - ΔW*|_F = 0.00001115
Epoch 5000/5000: Loss = 0.00000011, |ΔW_train - ΔW*|_F = 0.00099456
--------------------
LoRA 训练完成。


In [ ]:
print(B)
print(u_np)

Parameter containing:
tensor([[0.6374],
        [0.6374],
        [0.6374],
        [0.6374],
        [0.6374]], requires_grad=True)
[[0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]]


In [ ]:
print(A)
print(v_T_np)

Parameter containing:
tensor([[ 7.8443e-01,  7.3904e-04, -5.0642e-04, -5.5772e-06, -2.0283e-05]],
       requires_grad=True)
[[1. 0. 0. 0. 0.]]


In [ ]:
# --- 7. 最终结果分析 ---

# 训练得到的 Delta_W
Delta_W_train_final = B @ A * SCALE

# 计算误差
delta_w_error_final = linalg.norm(Delta_W_train_final - Delta_W_star, ord='fro')

print("--- 结果比较 ---")
print(f"目标更新 ΔW* 的 Frobenius 范数: {linalg.norm(Delta_W_star, ord='fro').item():.4f}")
print(f"训练后 ΔW_train 的 Frobenius 范数: {linalg.norm(Delta_W_train_final, ord='fro').item():.4f}")
print(f"ΔW 训练误差 (|ΔW_train - ΔW*|_F): {delta_w_error_final.item():.8f}")
print("-" * 20)

print("目标更新 ΔW* (解析解):")
print(Delta_W_star)
print("-" * 20)

print("训练得到的 ΔW_train (LoRA 训练解):")
print(Delta_W_train_final)
print("-" * 20)


--- 结果比较 ---
目标更新 ΔW* 的 Frobenius 范数: 1.1180
训练后 ΔW_train 的 Frobenius 范数: 1.1180
ΔW 训练误差 (|ΔW_train - ΔW*|_F): 0.00127730
--------------------
目标更新 ΔW* (解析解):
tensor([[0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.0000, 0.0000, 0.0000, 0.0000]])
--------------------
训练得到的 ΔW_train (LoRA 训练解):
tensor([[ 5.0000e-01,  4.7107e-04, -3.2280e-04, -3.5550e-06, -1.2929e-05],
        [ 5.0000e-01,  4.7107e-04, -3.2280e-04, -3.5550e-06, -1.2929e-05],
        [ 5.0000e-01,  4.7107e-04, -3.2280e-04, -3.5550e-06, -1.2929e-05],
        [ 5.0000e-01,  4.7107e-04, -3.2280e-04, -3.5550e-06, -1.2929e-05],
        [ 5.0000e-01,  4.7107e-04, -3.2280e-04, -3.5550e-06, -1.2929e-05]],
       grad_fn=<MulBackward0>)
--------------------
结论：训练结果与目标之间存在一定误差，可能需要更多训练或调整学习率。


In [ ]:

import torch.linalg as linalg

# 确保 Delta_W_star 仍在内存中。如果之前运行的 Cell 还在，这个变量就存在。

# --- 目标更新矩阵 ΔW* 的奇异值分解 ---
# torch.linalg.svd 返回 U, S (奇异值向量), Vh (V的共轭转置，即 V^T)
U, S, Vh = linalg.svd(Delta_W_star)

# 1. 奇异值 S (SVD 中的中间对角矩阵元素)
sigma_1 = S[0].item()
print("1. 奇异值 (σ₁):")
print(f"σ₁ ≈ {sigma_1:.4f}") # 预计为 1.1180

# 2. 单位左奇异向量 U₁ (相当于 LoRA 中的 B 的方向)
U1 = U[:, 0].reshape(5, 1)
print("\n2. 单位左奇异向量 U₁ (模长为 1 的 B 向量):")
print(U1.numpy().round(4))

# 3. 单位右奇异向量 V₁ᵀ (相当于 LoRA 中的 A 的方向)
V1_T = Vh[0, :].reshape(1, 5)
print("\n3. 单位右奇异向量 V₁ᵀ (模长为 1 的 A 向量):")
print(V1_T.numpy().round(4))

# 4. 验证奇异值是如何被吸收的
# 在 SVD 中： ΔW* = U₁ @ σ₁ @ V₁ᵀ
# 在 LoRA 中： ΔW* = B @ A
B_SVD = U1 * sigma_1  # 让 B 吸收奇异值
A_SVD = V1_T          # 让 A 保持单位化

print("\n4. SVD 结果验证 (B 吸收 σ₁):")
print(f"SVD 后的 B (U₁ * σ₁): \n{B_SVD.numpy().round(4)}")
print(f"SVD 后的 A (V₁ᵀ): \n{A_SVD.numpy().round(4)}")

1. 奇异值 (σ₁):
σ₁ ≈ 1.1180

2. 单位左奇异向量 U₁ (模长为 1 的 B 向量):
[[-0.4472]
 [-0.4472]
 [-0.4472]
 [-0.4472]
 [-0.4472]]

3. 单位右奇异向量 V₁ᵀ (模长为 1 的 A 向量):
[[-1. -0. -0. -0. -0.]]

4. SVD 结果验证 (B 吸收 σ₁):
SVD 后的 B (U₁ * σ₁): 
[[-0.5]
 [-0.5]
 [-0.5]
 [-0.5]
 [-0.5]]
SVD 后的 A (V₁ᵀ): 
[[-1. -0. -0. -0. -0.]]
